In [1]:
# Imports dan paths

import pandas as pd, numpy as np, optuna, mlflow, xgboost as xgb
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
from pathlib import Path
import json, joblib

In [ ]:
# Load data
DAILY_PARQUET = Path("../data/processed/daily_features.parquet")
df_daily = pd.read_parquet(DAILY_PARQUET)
df_daily.head()

In [ ]:
df_daily.tail()

In [ ]:
df_daily["date_local"] = pd.to_datetime(df_daily["date_local"])

In [ ]:
### Feature engineering – lags, rolling means, DOY encodings
LAGS  = [1, 3]
ROLLS = [3, 7]

df = df_daily.sort_values(["station_id", "date_local"]).set_index("date_local")

for col in ["temp_13LT_C", "rh_avg_pc", "wind_avg_kmh",
            "qff_avg_hPa", "rain_mm", "evap_mm"]:
    for k in LAGS:
        df[f"{col}_lag{k}"] = df.groupby("station_id")[col].shift(k)
    for w in ROLLS:
        df[f"{col}_roll{w}"] = (
            df.groupby("station_id")[col]
              .rolling(w, min_periods=1).mean()
              .droplevel(0)
        )

df["doy_sin"] = np.sin(2*np.pi*df.index.dayofyear / 365.25)
df["doy_cos"] = np.cos(2*np.pi*df.index.dayofyear / 365.25)
df["month"]   = df.index.month

# drop first-lag rows that now contain NaN
df = df.dropna(subset=[c for c in df.columns if "lag" in c]).reset_index()
df.head()


In [ ]:
df.to_csv("/home/rzby/ffmc_dc/data/processed/feature_engineering.csv", index=False)

In [ ]:
###  NEW – station-stratified random split

from sklearn.utils import shuffle

test_frac = 0.20
test_idx  = (
    df.groupby('station_id', group_keys=False)
      .apply(lambda g: g.sample(frac=test_frac, random_state=42))
      .index
)

train_idx = df.index.difference(test_idx)

train = df.loc[train_idx].copy()
test  = df.loc[test_idx].copy()

PRED_COLS = [c for c in df.columns if c not in ('station_id', 'ffmc', 'date_local')]
X_train, y_train = train[PRED_COLS], train['ffmc']
X_test,  y_test  = test[PRED_COLS],  test['ffmc']
print(f"Train rows: {len(train):,}  |  Test rows: {len(test):,}")

print(y_train[:10])
print(y_test[:10])


In [ ]:
### Optuna + MLflow tuning – find best XGBoost hyper-params
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("ffmc_xgb")

def objective(trial):
    params = {
        "n_estimators"     : trial.suggest_int("n_estimators", 300, 1200),
        "max_depth"        : trial.suggest_int("max_depth", 3, 10),
        "learning_rate"    : trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample"        : trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree" : trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "random_state"     : 42,
        "n_jobs"           : -1,
    }
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))

    with mlflow.start_run():
        mlflow.log_params(params)
        mlflow.log_metric("rmse", rmse)
        mlflow.xgboost.log_model(model, "model")
    return rmse

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=40, show_progress_bar=True)

best_rmse = study.best_value
print("Best RMSE:", best_rmse)


In [ ]:
### Retrieve best model & evaluate RQ 1 metrics
best_run = mlflow.search_runs(order_by=["metrics.rmse"], max_results=1).iloc[0]
model_uri = f"runs:/{best_run.run_id}/model"
best_model = mlflow.xgboost.load_model(model_uri)

test["pred_xgb"] = best_model.predict(X_test)

overall = {
    "RMSE": np.sqrt(mean_squared_error(y_test, test["pred_xgb"])),
    "MAE" : mean_absolute_error(y_test, test["pred_xgb"]),
    "R2"  : r2_score(y_test, test["pred_xgb"]),
}
print("=== Overall test metrics ===")
for k, v in overall.items():
    print(f"{k:4s}: {v:6.3f}")

per_station = (
    test.groupby("station_id")
        .apply(lambda g: pd.Series({
            "RMSE": np.sqrt(mean_squared_error(g["ffmc"], g["pred_xgb"])),
            "MAE" : mean_absolute_error(g["ffmc"], g["pred_xgb"]),
            "R2"  : r2_score(g["ffmc"], g["pred_xgb"]),
        }))
        .sort_index()
)
per_station


In [ ]:
per_station.to_csv("/home/rzby/ffmc_dc/reports/xgb_per_station.csv")

In [ ]:
### ⬛ ANN – build, train, predict  (CELL 5-A)

from tensorflow import keras
from sklearn.preprocessing import StandardScaler

# 1. scale predictors
scaler_ann = StandardScaler().fit(X_train)
Xtr_ann = scaler_ann.transform(X_train)
Xte_ann = scaler_ann.transform(X_test)

# 2. build model
ann = keras.Sequential([
    keras.layers.Dense(128, activation="relu", input_shape=(Xtr_ann.shape[1],)),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(1)         # linear output
])
ann.compile(loss="mse", optimizer=keras.optimizers.Adam(1e-3), metrics=["mae"])

# 3. train with early stopping
callback = keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)
history = ann.fit(Xtr_ann, y_train,
                  validation_split=0.2,
                  epochs=300,
                  batch_size=128,
                  verbose=0,
                  callbacks=[callback])

# 4. predict
test["pred_ann"] = ann.predict(Xte_ann, verbose=0).ravel()


In [ ]:
### ⬛ SVM (RBF) – scale, tune C & γ quickly, predict  (CELL 5-B)

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV

pipe = make_pipeline(StandardScaler(), SVR(kernel="rbf"))

param_grid = {
    "svr__C":      [1, 10, 100],
    "svr__gamma":  ["scale", 0.01, 0.1],
    "svr__epsilon":[0.1, 0.2],
}
grid = GridSearchCV(pipe, param_grid, cv=3, scoring="neg_root_mean_squared_error", n_jobs=-1)
grid.fit(X_train, y_train)

print("Best SVM params:", grid.best_params_)
test["pred_svm"] = grid.best_estimator_.predict(X_test)


In [ ]:
### ▶▶ Collect overall & per-station metrics for all three models  (CELL 6 updated)

def metric_dict(y_true, y_hat):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_hat)),
        "MAE" : mean_absolute_error(y_true, y_hat),
        "R2"  : r2_score(y_true, y_hat),
    }

overall = pd.DataFrame({
    "XGB" : metric_dict(y_test, test["pred_xgb"]),
    "ANN" : metric_dict(y_test, test["pred_ann"]),
    "SVM" : metric_dict(y_test, test["pred_svm"]),
}).T
print("=== Overall test-set metrics ===")
overall


In [ ]:
overall.to_csv("/home/rzby/ffmc_dc/reports/overall_results.csv")

In [ ]:
test[["ffmc", "pred", "pred_ann", "pred_svm"]].to_csv(Path("../reports/output_table.csv"), index=False)

In [ ]:
### Per-station RMSE for each model  (CELL 6-bis)

per_station_all = (
    test.groupby("station_id")
        .apply(lambda g: pd.Series({
            "XGB_RMSE": np.sqrt(mean_squared_error(g["ffmc"], g["pred_xgb"])),
            "ANN_RMSE": np.sqrt(mean_squared_error(g["ffmc"], g["pred_ann"])),
            "SVM_RMSE": np.sqrt(mean_squared_error(g["ffmc"], g["pred_svm"])),
        }))
)
per_station_all


In [ ]:
per_station_all.to_csv("/home/rzby/ffmc_dc/reports/rmse_per_station.csv")

In [ ]:
# per station MAE
per_station_mae = (
    test.groupby("station_id")
        .apply(lambda g: pd.Series({
            "XGB_MAE": mean_absolute_error(g["ffmc"], g["pred_xgb"]),
            "ANN_MAE": mean_absolute_error(g["ffmc"], g["pred_ann"]),
            "SVM_MAE": mean_absolute_error(g["ffmc"], g["pred_svm"]),
        }))
)
per_station_mae

In [ ]:
per_station_mae.to_csv("/home/rzby/ffmc_dc/reports/mae_per_station.csv")

In [ ]:
# per station R2
per_station_r2 = (
    test.groupby("station_id")
        .apply(lambda g: pd.Series({
            "XGB_R2": r2_score(g["ffmc"], g["pred_xgb"]),
            "ANN_R2": r2_score(g["ffmc"], g["pred_ann"]),
            "SVM_R2": r2_score(g["ffmc"], g["pred_svm"]),
        }))
)
per_station_r2

In [ ]:
per_station_r2.to_csv("/home/rzby/ffmc_dc/reports/r2_per_station.csv")

In [ ]:
### Visual 3 – overall RMSE comparison  (CELL 7-bis)

overall.RMSE.plot.bar(rot=0, figsize=(4,3))
plt.ylabel("RMSE"); plt.title("Model comparison – overall RMSE")
plt.tight_layout(); plt.savefig(Path("../reports/figures/rmse_comparison.png"))
plt.show()

In [ ]:
### Visual 4 – per-station RMSE heatmap  (CELL 8-bis)
import seaborn as sns

plt.figure(figsize=(6,5))
sns.heatmap(per_station_all.filter(like="_RMSE").T,
            annot=True, fmt=".2f", cmap="YlOrRd")
plt.title("RMSE by station & model"); plt.xlabel("WMO station"); plt.ylabel("")
plt.tight_layout()
plt.savefig(Path("../reports/figures/rmse_station_comparison.png"))
plt.show()


In [ ]:
### Visual 1 – prediction vs observation scatter (CELL 7)
plt.figure(figsize=(5,5))
plt.scatter(test["ffmc"], test["pred"], s=10, alpha=0.5)
plt.plot([60,100], [60,100], "k--")
plt.xlabel("Observed FFMC"); plt.ylabel("Predicted FFMC")
plt.title("Overall test set – XGBoost")
plt.grid(True); plt.tight_layout()
plt.savefig(Path("../reports/figures/scatter_xgboost.png"))
plt.show()


In [ ]:
### RQ 2 – SHAP feature importance  (CELL 9)
import shap
sample = df.sample(6000, random_state=0)   # speed; adjust if GPU available
X_sample = sample[PRED_COLS]

explainer = shap.Explainer(best_model)
shap_values = explainer(X_sample, check_additivity=False)

shap.summary_plot(shap_values, X_sample, show=False)
plt.title("SHAP summary – key FFMC drivers")
plt.tight_layout()
plt.savefig(Path("../reports/figures/feature_xgboost.png"))
plt.show()


In [ ]:
# DUMP BEST MODEL
import json, joblib, numpy as np, pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

MODELS_DIR = Path("/home/rzby/ffmc_dc/models")
print(MODELS_DIR)
REPORTS_DIR = Path("/home/rzby/ffmc_dc/reports")
FIG_DIR = REPORTS_DIR / "figures"
OUT_DIR = Path("/home/rzby/ffmc_dc/reports")

PRED_COLS = [c for c in df.columns if c not in ("station_id","ffmc","date_local")]

meta = {
    "best_model": "XGB",
    "pred_cols": PRED_COLS
}

joblib.dump(best_model, MODELS_DIR / "best_model_xgb.pkl")
meta["path"] = str(MODELS_DIR / "best_model_xgb.pkl")
meta["kind"] = "xgboost"

with open(MODELS_DIR / "best_model_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved:", meta)

In [2]:
# Predict Last 3 Days and Prepare for Mapping

MODELS_DIR = Path("/home/rzby/ffmc_dc/models")
OUT_DIR = Path("/home/rzby/ffmc_dc/reports")
# Load features with coordinates
features = pd.read_csv("/home/rzby/ffmc_dc/data/processed/feature_engineering.csv")
features['station_id'] = features['station_id'].astype(str)
features['date_local'] = pd.to_datetime(features['date_local'])

# Ensure lon/lat are present; if not, read from `data/stations.csv`
if not {"lon","lat"}.issubset(features.columns):
    stations_csv = Path("/home/rzby/ffmc_dc/data/stations.csv")
    if not stations_csv.exists():
        raise FileNotFoundError(
            "No lon/lat in features and stations.csv not found.\n"
            "Create data/stations.csv with columns: station_id,lon,lat[,name]"
        )
    stations = pd.read_csv(stations_csv)
    stations['station_id'] = stations['station_id'].astype(str)
    features = features.merge(stations, on="station_id", how="left")

# pick last 3 *available* dates
last_date = pd.to_datetime("2023-12-31 00:00:00+00:00")
target_dates = pd.date_range(last_date - pd.Timedelta(days=3286), last_date, freq="D")
print("Target dates:", target_dates.date.tolist())

# load best artefacts back (to make this cell standalone)
with open(MODELS_DIR / "best_model_meta.json") as f:
    meta = json.load(f)

kind = meta["kind"]
pred_cols = meta["pred_cols"]

if kind == "xgboost":
    model = joblib.load(meta["path"])
elif kind == "ann":
    from tensorflow import keras
    model = keras.models.load_model(meta["path"])
    scaler_ann = joblib.load(meta["scaler"])
elif kind == "svm":
    model = joblib.load(meta["path"])

pred_rows = []
for d in target_dates:
    sub = features.loc[features["date_local"] == d].copy()
    if sub.empty:
        continue
    Xd = sub[pred_cols].copy()
    if kind == "ann":
        Xd = scaler_ann.transform(Xd)
        yhat = model.predict(Xd, verbose=0).ravel()
    else:
        yhat = model.predict(Xd)
    sub["ffmc_pred"] = yhat
    pred_rows.append(sub[["station_id","date_local","lon","lat","ffmc","ffmc_pred"]])

pred3 = pd.concat(pred_rows, ignore_index=True)
pred3.to_csv(OUT_DIR / "pred_all_points.csv", index=False)
pred3.head()


Target dates: [datetime.date(2015, 1, 1), datetime.date(2015, 1, 2), datetime.date(2015, 1, 3), datetime.date(2015, 1, 4), datetime.date(2015, 1, 5), datetime.date(2015, 1, 6), datetime.date(2015, 1, 7), datetime.date(2015, 1, 8), datetime.date(2015, 1, 9), datetime.date(2015, 1, 10), datetime.date(2015, 1, 11), datetime.date(2015, 1, 12), datetime.date(2015, 1, 13), datetime.date(2015, 1, 14), datetime.date(2015, 1, 15), datetime.date(2015, 1, 16), datetime.date(2015, 1, 17), datetime.date(2015, 1, 18), datetime.date(2015, 1, 19), datetime.date(2015, 1, 20), datetime.date(2015, 1, 21), datetime.date(2015, 1, 22), datetime.date(2015, 1, 23), datetime.date(2015, 1, 24), datetime.date(2015, 1, 25), datetime.date(2015, 1, 26), datetime.date(2015, 1, 27), datetime.date(2015, 1, 28), datetime.date(2015, 1, 29), datetime.date(2015, 1, 30), datetime.date(2015, 1, 31), datetime.date(2015, 2, 1), datetime.date(2015, 2, 2), datetime.date(2015, 2, 3), datetime.date(2015, 2, 4), datetime.date(2015

,station_id,date_local,lon,lat,ffmc,ffmc_pred
0,96595,2015-01-04 00:00:00+00:00,114.53,-0.56,60.330249,60.456619
1,96645,2015-01-04 00:00:00+00:00,111.66,-2.73,66.682773,66.717758
2,96651,2015-01-04 00:00:00+00:00,112.93,-2.55,81.911389,82.472794
3,96653,2015-01-04 00:00:00+00:00,114.90,-1.67,67.050474,66.980278
4,96655,2015-01-04 00:00:00+00:00,113.95,-2.22,70.015149,70.021065


In [3]:
pred3['month_name'] = pred3['date_local'].dt.month_name

In [4]:
pred3.to_csv(OUT_DIR / "pred_all_points.csv", index=False)

In [ ]:
monthly_avg = pred3.groupby(['station_id', pd.Grouper(key='date_local', freq='M')])['ffmc_pred'].mean().reset_index()

print(monthly_avg)

In [ ]:
monthly_avg['month'] = monthly_avg['date_local'].dt.month
monthly_avg = monthly_avg.drop('date_local', axis=1)

In [ ]:
monthly_avg.to_csv(OUT_DIR / "monthly_avg.csv", index=False)

In [ ]:
pred3.to_csv(f'{REPORTS_DIR}/mapping_output.csv', index=False)

In [ ]:
# MAKE POINTS MAP
import matplotlib.pyplot as plt

# Optional: province boundary
PROV_SHP = None  # e.g., "data/geo/kalimantan_tengah.shp"

def plot_points(df_day, date_label, out_png):
    plt.figure(figsize=(6,6))
    if PROV_SHP:
        import geopandas as gpd
        prov = gpd.read_file(PROV_SHP).to_crs(4326)
        prov.plot(edgecolor="black", facecolor="none")
    sc = plt.scatter(df_day["lon"], df_day["lat"],
                     c=df_day["ffmc_pred"], s=120, edgecolor="k")
    plt.colorbar(sc, label="Predicted FFMC")
    for _, r in df_day.iterrows():
        plt.text(r["lon"]+0.03, r["lat"]+0.03, str(r["station_id"]), fontsize=8)
    plt.title(f"Predicted FFMC – {date_label}")
    plt.xlabel("Longitude"); plt.ylabel("Latitude")
    # set a reasonable frame around the points
    xmin, xmax = df_day["lon"].min()-0.5, df_day["lon"].max()+0.5
    ymin, ymax = df_day["lat"].min()-0.5, df_day["lat"].max()+0.5
    plt.xlim(xmin, xmax); plt.ylim(ymin, ymax)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.show()

for d, g in pred3.groupby("date_local"):
    out_png = FIG_DIR / f"ffmc_pred_points_{pd.to_datetime(d).date()}.png"
    plot_points(g, pd.to_datetime(d).date(), out_png)
    print("Saved:", out_png)
